In [1]:
from temgym_core.components import Component, Lens
from temgym_core.propagator import FreeSpaceParaxial
from temgym_core.ray import Ray

import sympy as sym

In [2]:
def test_propagate(ray, dist):
    pr = FreeSpaceParaxial.propagate(ray, dist)
    return pr

In [3]:
x, y, dx, dy, z, S, one, f = sym.symbols('x y dx dy z S one f')

In [4]:
ray = Ray(x, y, dx, dy, z, pathlength=S, _one=one)

In [5]:
test_propagate(ray, 1.)

Ray(x=1.0*dx + x, y=1.0*dy + y, dx=dx, dy=dy, z=z + 1.0, pathlength=S + 0.5*dx**2 + 0.5*dy**2 + 1.0, _one=one)

In [6]:
z_0, z_1 = sym.symbols('z_0 z_1') # distance

In [7]:
ray_prop = test_propagate(ray, z_0)
ray_prop

Ray(x=dx*z_0 + x, y=dy*z_0 + y, dx=dx, dy=dy, z=z + z_0, pathlength=S + z_0*(0.5*dx**2 + 0.5*dy**2 + 1), _one=one)

In [8]:
def test_lens(z, f, ray):
    res_tmp = Lens(z, f).__call__(ray=ray)
    return res_tmp

In [9]:
ray_prop_lens = test_lens(z, f, ray_prop)

In [10]:
ray_prop_lens

Ray(x=dx*z_0 + x, y=dy*z_0 + y, dx=dx + (-dx*z_0 - x)/f, dy=dy + (-dy*z_0 - y)/f, z=z + z_0, pathlength=S + z_0*(0.5*dx**2 + 0.5*dy**2 + 1) - ((dx*z_0 + x)**2 + (dy*z_0 + y)**2)/(2*f), _one=1.0*one)

In [11]:
# sym.diff(res, x)

In [12]:
# sym.diff(res, y)

In [13]:
ray_prop_lens_prop = test_propagate(ray_prop_lens, z_1)

In [14]:
ray_prop_lens_prop

Ray(x=dx*z_0 + x + z_1*(dx + (-dx*z_0 - x)/f), y=dy*z_0 + y + z_1*(dy + (-dy*z_0 - y)/f), dx=dx + (-dx*z_0 - x)/f, dy=dy + (-dy*z_0 - y)/f, z=z + z_0 + z_1, pathlength=S + z_0*(0.5*dx**2 + 0.5*dy**2 + 1) + z_1*(0.5*(dx + (-dx*z_0 - x)/f)**2 + 0.5*(dy + (-dy*z_0 - y)/f)**2 + 1) - ((dx*z_0 + x)**2 + (dy*z_0 + y)**2)/(2*f), _one=1.0*one)

In [15]:
def ray_to_matrix(ray):
    list_ray = []
    for comp in ['x', 'y', 'dx', 'dy', '_one']:
        list_ray.append(ray.__dict__[comp])
    matrix_ray = sym.Matrix(list_ray)
    return matrix_ray

In [16]:
ray_matrix = ray_to_matrix(ray_prop_lens_prop)

In [17]:
ray_matrix

Matrix([
[dx*z_0 + x + z_1*(dx + (-dx*z_0 - x)/f)],
[dy*z_0 + y + z_1*(dy + (-dy*z_0 - y)/f)],
[                   dx + (-dx*z_0 - x)/f],
[                   dy + (-dy*z_0 - y)/f],
[                                1.0*one]])

In [18]:
ray_matrix.jacobian([x, y, dx, dy, one])

Matrix([
[1 - z_1/f,         0, z_0 + z_1*(1 - z_0/f),                     0,   0],
[        0, 1 - z_1/f,                     0, z_0 + z_1*(1 - z_0/f),   0],
[     -1/f,         0,             1 - z_0/f,                     0,   0],
[        0,      -1/f,                     0,             1 - z_0/f,   0],
[        0,         0,                     0,                     0, 1.0]])

In [19]:
def Lens_nonlin(z: float, focal_length: float, ray: Ray):
        f = focal_length

        x, y, dx, dy = ray.x, ray.y, ray.dx, ray.dy

        new_dx = -x**3 / f + dx
        new_dy = -y**3 / f + dy

        pathlength = ray.pathlength - (x**2 + y**2) / (2 * f)
        one = ray._one * 1.0

        return Ray(
            x=x, y=y, dx=new_dx, dy=new_dy, _one=one, pathlength=pathlength, z=ray.z
        )

In [20]:
from sympy.polys.ring_series import rs_series

In [21]:
def test_lens_nonlin(z, f, ray):
    res_tmp = Lens_nonlin(z, f, ray=ray)
    return res_tmp

In [22]:
ray_nonlin = test_lens_nonlin(z, f, ray_prop)
ray_prop_nonlin_prop = test_propagate(ray_nonlin, z_1)

In [23]:
ray_prop_nonlin_prop

Ray(x=dx*z_0 + x + z_1*(dx - (dx*z_0 + x)**3/f), y=dy*z_0 + y + z_1*(dy - (dy*z_0 + y)**3/f), dx=dx - (dx*z_0 + x)**3/f, dy=dy - (dy*z_0 + y)**3/f, z=z + z_0 + z_1, pathlength=S + z_0*(0.5*dx**2 + 0.5*dy**2 + 1) + z_1*(0.5*(dx - (dx*z_0 + x)**3/f)**2 + 0.5*(dy - (dy*z_0 + y)**3/f)**2 + 1) - ((dx*z_0 + x)**2 + (dy*z_0 + y)**2)/(2*f), _one=1.0*one)